# Collect Data

In [636]:
%pip install pandas numpy seaborn matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [637]:
import pandas as pd
import numpy as np

In [638]:
import time
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [639]:
df = pd.read_csv('final_data.csv')
df

,Idade,Petroleo,Agua,Gas,Gas_Royalties,Tempo_Producao,Grau_API,Densidade_GLP_Gas,PCS,Nome_Poco_ANP
0,4073,10597.4769,4800.2166,56.5923,1355.916,744,33.30,2.02232,40592.3314,3-BRSA-1024-SPS
1,3702,3245.1189,8032.5534,16.1271,386.394,744,33.30,2.02232,40592.3314,7-PRA-2-SPS
2,3563,0.0000,0.0000,0.0000,0.000,0,33.30,2.02232,40592.3314,7-BAN-2-SPS
3,4128,2116.3271,5439.4737,25.6074,613.536,714,33.30,2.02232,40592.3314,9-BRSA-999-SPS
4,3609,1978.3072,5187.5167,16.1911,387.928,712,33.30,2.02232,40592.3314,7-BAN-1-SPS
...,...,...,...,...,...,...,...,...,...,...
24521,14311,221.0590,4176.8396,0.8269,25.633,580,22.86,2.01722,40657.3157,7-PG-19D-RJS
24522,14562,253.8190,1296.9182,1.2978,40.232,661,22.86,2.01722,40657.3157,7-PG-12D-RJS
24523,14893,635.0981,1784.2771,1.9080,59.149,727,22.86,2.01722,40657.3157,7-PG-4-RJS
24524,14154,460.4175,4570.8658,1.3961,43.278,725,22.86,2.01722,40657.3157,7-PG-24D-RJS


In [640]:
from math import log

In [641]:
df['Agua_log'] = df['Agua'].apply(lambda x: log(x) if x > 0 else 0)
df['Gas_log'] = df['Gas'].apply(lambda x: log(x) if x > 0 else 0)
df['Idade2'] = df['Idade'].apply(lambda x: x**2)
df

,Idade,Petroleo,Agua,Gas,Gas_Royalties,Tempo_Producao,Grau_API,Densidade_GLP_Gas,PCS,Nome_Poco_ANP,Agua_log,Gas_log,Idade2
0,4073,10597.4769,4800.2166,56.5923,1355.916,744,33.30,2.02232,40592.3314,3-BRSA-1024-SPS,8.476416,4.035873,16589329
1,3702,3245.1189,8032.5534,16.1271,386.394,744,33.30,2.02232,40592.3314,7-PRA-2-SPS,8.991258,2.780501,13704804
2,3563,0.0000,0.0000,0.0000,0.000,0,33.30,2.02232,40592.3314,7-BAN-2-SPS,0.000000,0.000000,12694969
3,4128,2116.3271,5439.4737,25.6074,613.536,714,33.30,2.02232,40592.3314,9-BRSA-999-SPS,8.601438,3.242881,17040384
4,3609,1978.3072,5187.5167,16.1911,387.928,712,33.30,2.02232,40592.3314,7-BAN-1-SPS,8.554010,2.784462,13024881
...,...,...,...,...,...,...,...,...,...,...,...,...,...
24521,14311,221.0590,4176.8396,0.8269,25.633,580,22.86,2.01722,40657.3157,7-PG-19D-RJS,8.337310,-0.190072,204804721
24522,14562,253.8190,1296.9182,1.2978,40.232,661,22.86,2.01722,40657.3157,7-PG-12D-RJS,7.167746,0.260671,212051844
24523,14893,635.0981,1784.2771,1.9080,59.149,727,22.86,2.01722,40657.3157,7-PG-4-RJS,7.486769,0.646056,221801449
24524,14154,460.4175,4570.8658,1.3961,43.278,725,22.86,2.01722,40657.3157,7-PG-24D-RJS,8.427458,0.333683,200335716


In [642]:
target_column = 'Petroleo'
ignore_columns = [
    'Nome_Poco_ANP', 'PCS'
]

In [643]:
sample_df = df.sample(frac=1, random_state=42)
sample_df.drop_duplicates(inplace=True)
columns = [col for col in sample_df.columns if col not in ignore_columns]
for col in columns:
    Q1 = sample_df[col].quantile(0.25)
    Q3 = sample_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    sample_df = sample_df[(sample_df[col] >= lower_bound) & (sample_df[col] <= upper_bound)]

sample_df

,Idade,Petroleo,Agua,Gas,Gas_Royalties,Tempo_Producao,Grau_API,Densidade_GLP_Gas,PCS,Nome_Poco_ANP,Agua_log,Gas_log,Idade2
20903,6504,69.0258,1838.2138,3.9229,121.601,741,30.23,2.15079,35009.5128,7-SER-21DA-RNS,7.516550,1.366831,42302016
11075,622,5072.2519,60.0237,10.7341,321.711,637,13.42,2.10547,37502.5301,7-PRG-80HP-RJS,4.094739,2.373426,386884
4252,6468,1183.8165,1994.3474,8.9453,277.056,733,24.00,2.04402,40091.4179,7-JUB-6HB-ESS,7.598072,2.191128,41835024
7615,5189,258.2740,977.8236,1.7193,52.995,726,18.10,2.12516,36539.5523,7-ARGO-2H-ESS,6.885329,0.541917,26925721
5359,5591,218.1283,1941.8352,0.4434,13.303,714,30.23,2.12184,36715.5245,7-SER-20D-RNS,7.571389,-0.813283,31259281
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6265,5463,0.0000,0.0000,0.0000,0.000,0,18.10,2.00677,39989.7428,7-ABA-2D-ESS,0.000000,0.000000,29844369
11964,488,102.3119,5.0487,3.9909,56.696,62,30.20,2.00144,38798.0539,7-TUP-128D-RJS,1.619131,1.384017,238144
5390,348,578.2746,0.5789,9.2298,264.040,61,24.70,2.01939,40444.9634,7-MLL-90H-RJS,-0.546626,2.222437,121104
860,2638,132.0420,3.2312,5.2944,76.693,619,30.20,2.00144,38798.0539,7-LL-80DB-RJS,1.172854,1.666650,6959044


In [ ]:
sample_df['Nome_Poco_ANP'].value_counts().to_dict()

pocos_teste = dict()

while sum(pocos_teste.values()) < 0.2 * len(sample_df):
    poco = sample_df['Nome_Poco_ANP'].sample(1).iloc[0]
    if poco not in pocos_teste:
        pocos_teste[poco] = sample_df[sample_df['Nome_Poco_ANP'] == poco].shape[0]

pocos_teste = pocos_teste.keys()

In [ ]:
X = sample_df.drop([target_column], axis=1)
y = sample_df[[target_column] + ignore_columns]

X_train = X[~X['Nome_Poco_ANP'].isin(pocos_teste)].drop(ignore_columns, axis=1)
y_train = y[~y['Nome_Poco_ANP'].isin(pocos_teste)].drop(ignore_columns, axis=1)

X_test = X[X['Nome_Poco_ANP'].isin(pocos_teste)].drop(ignore_columns, axis=1)
y_test = y[y['Nome_Poco_ANP'].isin(pocos_teste)].drop(ignore_columns, axis=1)

In [646]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Evaluate Methods

In [647]:
results_df = pd.DataFrame()

def evaluate_and_record(model_name, model, X_train, y_train, X_test, y_test, best_params=None):
    global results_df
    
    start_time = time.time()
    model.fit(X_train, y_train)
    end_time = time.time()
    training_time = end_time - start_time
    
    y_pred_treino = model.predict(X_train)
    y_pred_teste = model.predict(X_test)
    
    rmse_treino = np.sqrt(mean_squared_error(y_train, y_pred_treino))
    r2_treino = r2_score(y_train, y_pred_treino)
    mae_treino = mean_absolute_error(y_train, y_pred_treino)
    
    rmse_teste = np.sqrt(mean_squared_error(y_test, y_pred_teste))
    r2_teste = r2_score(y_test, y_pred_teste)
    mae_teste = mean_absolute_error(y_test, y_pred_teste)
    
    new_row = {
        'Modelo': model_name,
        'R2 Treino': r2_treino,
        'R2 Teste': r2_teste,
        'RMSE Treino': rmse_treino,
        'RMSE Teste': rmse_teste,
        'MAE Treino': mae_treino,
        'MAE Teste': mae_teste,
        'Tempo (segundos)': training_time,
        'Hyperparametros': str(best_params) if best_params else 'N/A'
    }
    
    results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)
    return model

In [648]:
def record_gridsearch_details(grid_search_object, model_prefix):
    global results_df
    
    results = pd.DataFrame(grid_search_object.cv_results_)
    
    for index, row in results.iterrows():
        mean_r2_cv = row['mean_test_score']
        
        mean_fit_time = row['mean_fit_time']
        
        model_name_detail = f"{model_prefix} - Combinacao {index + 1}"
        
        new_row = {
            'Modelo': model_name_detail,
            'R2 Teste': mean_r2_cv,
            'RMSE Teste': np.nan,
            'Tempo (s)': mean_fit_time,
            'Detalhes': f"Combin. Testada (Rank: {row['rank_test_score']})",
            'Hyperparametros': str(row['params'])
        }
        
        results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)

# LinearRegression

In [649]:
model = LinearRegression()
evaluate_and_record("Regressao Linear", model, X_train_scaled, y_train, X_test_scaled, y_test)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1, 10)","[[-692.94, 51.67, 119.29,..., 113.34, 216.43, 464.55]]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.","ndarray[float64](1,)",[941.56]
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,10
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,10
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](10,)","[172.64,157.78,122.45,..., 45.08, 38.45, 19.01]"


# Ridge

In [650]:
alphas = np.logspace(-4, 2, 100)

scoring_metric = 'r2'
ridge_model = Ridge(random_state=42)
grid_search_ridge = GridSearchCV(
    estimator=ridge_model,
    param_grid={'alpha': alphas},
    scoring=scoring_metric,
    cv=5,
    verbose=0,
    n_jobs=-1
)

grid_search_ridge.fit(X_train_scaled, y_train)

best_alpha_ridge = grid_search_ridge.best_params_['alpha']
best_ridge_model = grid_search_ridge.best_estimator_

In [651]:
evaluate_and_record('Ridge Otimo', best_ridge_model, X_train_scaled, y_train, X_test_scaled, y_test, best_alpha_ridge)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",np.float64(2.009233002565046)
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary <random_state>` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",42
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 

# Lasso

In [652]:
lasso_model = Lasso(random_state=42, max_iter=2000) 
grid_search_lasso = GridSearchCV(
    estimator=lasso_model,
    param_grid={'alpha': alphas},
    scoring=scoring_metric,
    cv=5,
    verbose=0,
    n_jobs=-1
)

grid_search_lasso.fit(X_train_scaled, y_train)

best_alpha_lasso = grid_search_lasso.best_params_['alpha']
best_lasso_model = grid_search_lasso.best_estimator_

In [653]:
evaluate_and_record('Lasso Otimo', best_lasso_model, X_train_scaled, y_train, X_test_scaled, y_test, best_alpha_lasso)

,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",np.float64(0....4802310718645)
,"max_iter max_iter: int, default=1000The maximum number of iterations.",2000
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary <warm_start>`.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'
Name,Type,Value


# Random Forest

In [655]:
from sklearn.ensemble import RandomForestRegressor

In [656]:
model_rf = RandomForestRegressor(max_depth=20, min_samples_leaf=4, min_samples_split=6, n_estimators=1000, random_state=42)
model_rf.fit(X_train_scaled, y_train)

print(f"Random Forest - R2 Teste: {r2_score(y_test, model_rf.predict(X_test_scaled)):.4f}, RMSE Teste: {np.sqrt(mean_squared_error(y_test, model_rf.predict(X_test_scaled))):.4f}")

/home/icarorodrigues/Downloads/Codigos Mestrado/MMQ_MSC_Icaro/venv/lib/python3.12/site-packages/sklearn/base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Random Forest - R2 Teste: 0.8146, RMSE Teste: 376.8633


In [558]:
rf_model = RandomForestRegressor(random_state=42)
grid_search_rf = GridSearchCV(
    estimator=rf_model,
    param_grid={
        'n_estimators': [100, 200, 300, 400, 500, 600],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    },
    scoring=scoring_metric,
    cv=3,
    verbose=0,
    n_jobs=-1
)

grid_search_rf.fit(X_train_scaled, y_train)

best_n_estimators_rf = grid_search_rf.best_params_['n_estimators']
best_max_depth_rf = grid_search_rf.best_params_['max_depth']
best_min_samples_split_rf = grid_search_rf.best_params_['min_samples_split']
best_min_samples_leaf_rf = grid_search_rf.best_params_['min_samples_leaf']
best_rf_model = grid_search_rf.best_estimator_

ValueError: 
All the 216 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
72 fits failed with the following error:
Traceback (most recent call last):
  File "/home/icarorodrigues/Downloads/Codigos Mestrado/MMQ_MSC_Icaro/venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/icarorodrigues/Downloads/Codigos Mestrado/MMQ_MSC_Icaro/venv/lib/python3.12/site-packages/sklearn/base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/icarorodrigues/Downloads/Codigos Mestrado/MMQ_MSC_Icaro/venv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py", line 396, in fit
    y = np.ascontiguousarray

In [ ]:
evaluate_and_record('Random Forest Otimo', best_rf_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_rf.best_params_)

,n_estimators,600
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


# Gradient Boosting

In [ ]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from xgboost import XGBRegressor

In [ ]:
xgb_model = XGBRegressor(objective='reg:squarederror', random_state=42)
grid_search_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid={
        'n_estimators': [100, 300, 500, 700],
        'learning_rate': [0.01, 0.1, 0.3],
        'max_depth': [3, 5, 7],
        'subsample': [0.7, 1.0],
    },
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)
grid_search_xgb.fit(X_train_scaled, y_train)

best_n_estimators_xgb = grid_search_xgb.best_params_['n_estimators']
best_max_depth_xgb = grid_search_xgb.best_params_['max_depth']
best_learning_rate_xgb = grid_search_xgb.best_params_['learning_rate']
best_subsample_xgb = grid_search_xgb.best_params_['subsample']
best_xgb_model = grid_search_xgb.best_estimator_

Fitting 3 folds for each of 72 candidates, totalling 216 fits


In [ ]:
evaluate_and_record('Gradient Boosting Otimo', best_xgb_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_xgb.best_params_)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


# KNN Regression

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

In [ ]:
knn_model = KNeighborsRegressor()
grid_search_knn = GridSearchCV(
    estimator=knn_model,
    param_grid={
        'n_neighbors': [3, 5, 7, 9],
        'weights': ['uniform', 'distance'],
        'p': [1, 2]
    },
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=0,
    n_jobs=-1
)

grid_search_knn.fit(X_train_scaled, y_train)
best_n_neighbors_knn = grid_search_knn.best_params_['n_neighbors']
best_weights_knn = grid_search_knn.best_params_['weights']
best_p_knn = grid_search_knn.best_params_['p']
best_knn_model = grid_search_knn.best_estimator_

In [ ]:
evaluate_and_record('KNN Otimo', best_knn_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_knn.best_params_)

,n_neighbors,5
,weights,'distance'
,algorithm,'auto'
,leaf_size,30
,p,1
,metric,'minkowski'
,metric_params,None
,n_jobs,None


# SVR (Suport Vector Regression)

In [ ]:
from sklearn.svm import SVR

In [ ]:
svr_model = SVR()
grid_search_svr = GridSearchCV(
    estimator=svr_model,
    param_grid={
        'C': [1000, 2000, 3000, 4000, 5000],
        'epsilon': [0.01, 0.1, 1],
        'kernel': ['linear', 'rbf']
    },
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=0,
    n_jobs=-1
)

grid_search_svr.fit(X_train_scaled, y_train)

best_C_svr = grid_search_svr.best_params_['C']
best_epsilon_svr = grid_search_svr.best_params_['epsilon']
best_kernel_svr = grid_search_svr.best_params_['kernel']
best_svr_model = grid_search_svr.best_estimator_

In [ ]:
evaluate_and_record('SVR Otimo', best_svr_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_svr.best_params_)

,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,5000
,epsilon,1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


# Neural Network

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

In [ ]:
mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(random_state=42, max_iter=2000))
])

mlp_param_grid = {
    'mlp__hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 100), (200, 100), 
                                (200, 200), (100, 100, 100)],
    'mlp__activation': ['relu', 'tanh'],
    'mlp__solver': ['adam', 'sgd'],
    'mlp__alpha': [0.0001, 0.001, 0.01, 0.1],
    'mlp__learning_rate': ['constant', 'adaptive']
}

grid_search_mlp = GridSearchCV(
    estimator=mlp_pipeline,
    param_grid=mlp_param_grid,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid_search_mlp.fit(X_train, y_train)
best_params_mlp = grid_search_mlp.best_params_
best_mlp_model = grid_search_mlp.best_estimator_

Fitting 3 folds for each of 224 candidates, totalling 672 fits


In [ ]:
evaluate_and_record('MLP Otimo', best_mlp_model, X_train, y_train, X_test, y_test, best_params_mlp)

MLP Regressor
Best parameters: {'mlp__activation': 'tanh', 'mlp__alpha': 0.1, 'mlp__hidden_layer_sizes': (200, 200), 'mlp__learning_rate': 'constant', 'mlp__solver': 'adam'}
RMSE: 94.8516 (Treino), 326.4406 (Teste)
MAE: 41.7225 (Treino), 149.3877 (Teste)
R2: 0.9946 (Treino), 0.9268 (Teste)


# GridSearch Details

In [ ]:
record_gridsearch_details(grid_search_ridge, "Ridge GridSearch")
record_gridsearch_details(grid_search_lasso, "Lasso GridSearch")
record_gridsearch_details(grid_search_rf, "Random Forest GridSearch")
record_gridsearch_details(grid_search_xgb, "XGBoost GridSearch")
record_gridsearch_details(grid_search_knn, "KNN GridSearch")
record_gridsearch_details(grid_search_svr, "SVR GridSearch")
record_gridsearch_details(grid_search_mlp, "MLP GridSearch")

In [ ]:
results_df

In [654]:
results_df.to_csv('model_comparison_results.csv', index=False)